![title.png](imgs/title-dsc-dach.png)

![section-breakpoint.png](imgs/section-breakpoint.png)

![objectives.png](imgs/objectives-berlin-2024.png)

![image.png](imgs/visual-breakpoint.png)

![pinecone-assistant-api.png](imgs/pinecone-assistant-api.png)

Let's see how it looks like in the end.

An out of the service that allows you to upload documents, ask questions, and receive responses that reference your documents.
More info and API examples: https://docs.pinecone.io/guides/assistant/understanding-assistant 

Pinecone Console: https://www.pinecone.io/

Papers to upload:
1. LLama 2: https://arxiv.org/pdf/2307.09288
2. LLama 3: https://arxiv.org/pdf/2407.21783 
3. SAM-2: https://arxiv.org/pdf/2408.00714 


How to chat with assistant programatically: https://docs.pinecone.io/guides/assistant/chat-with-assistant 

## Code sample
```python
# pip install --upgrade pinecone pinecone-plugin-assistant

from pinecone import Pinecone
from pinecone_plugins.assistant.models.chat import Message

pc = Pinecone(api_key="YOUR_API_KEY")

# Get your assistant.
assistant = pc.assistant.Assistant(
    assistant_name="example-assistant", 
)

# Chat with the assistant.
chat_context = [Message(content='What are key benefits of LLama 3?')]
response = assistant.chat_completions(messages=chat_context)
```

![image.png](imgs/visual-breakpoint.png)

## Notebook setup and dependency installation

In [37]:
!   pip install -qU \
    openai==1.49 \
    "pinecone[grpc]==5.3.1" \
    datasets==2.19 \
    pandas==2.2.2 \
    tqdm


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [63]:
from IPython.display import HTML, display, Markdown
from typing import Dict


def chunk_display_html(chunk: Dict[str, str]) -> str:
    html_template = """
<html>
<head>
<style>
    table {{
        font-family: arial, sans-serif;
        border-collapse: collapse;
        width: 100%;
    }}
    td, th {{
        border: 1px solid #dddddd;
        text-align: left;
        padding: 8px;
    }}
</style>
</head>
<body>
    <table>
        <tr>
            <th>Key</th>
            <th>Value</th>
        </tr>
        <tr>
            <td>Title</td>
            <td>{title}</td>
        </tr>
        <tr>
            <td>doi</td>
            <td>{doi}</td>
        </tr>
        <tr>
            <td>Chunk ID</td>
            <td>{chunk_id}</td>
        </tr>
        <tr>
            <td>chunk</td>
            <td>{chunk}</td>
        </tr>
        <tr>
            <td>id</td>
            <td>{id}</td>
        </tr>
        <tr>
            <td>Summary</td>
            <td>{summary}</td>
        </tr>
        <tr>
            <td>Source</td>
            <td>{source}</td>
        </tr>
        <tr>
            <td>Authors</td>
            <td>{authors}</td>
        </tr>
        <tr>
            <td>Categories</td>
            <td>{categories}</td>
        </tr>
        <tr>
            <td>Comment</td>
            <td>{comment}</td>
        </tr>
        <tr>
            <td>Journal Reference</td>
            <td>{journal_ref}</td>
        </tr>
        <tr>
            <td>Primary Category</td>
            <td>{primary_category}</td>
        </tr>
        <tr>
            <td>Published</td>
            <td>{published}</td>
        </tr>
        <tr>
            <td>Updated</td>
            <td>{updated}</td>
        </tr>
        <tr>
            <td>References</td>
            <td>{references}</td>
        </tr>
    </table>
</body>
</html>
"""

    # Format the HTML with the generated rows
    html_output = html_template.format(
        doi=chunk.get("doi", "N/A"),
        chunk_id=chunk.get("chunk-id", "N/A"),
        chunk=chunk.get("chunk", "N/A"),
        id=chunk.get("id", "N/A"),
        title=chunk.get("title", "N/A"),
        summary=chunk.get("summary", "N/A"),
        source=chunk.get("source", "N/A"),
        authors=chunk.get("authors", "N/A")[:500],
        categories=chunk.get("categories", "N/A"),
        comment=chunk.get("comment", "N/A"),
        journal_ref=chunk.get("journal_ref", "N/A"),
        primary_category=chunk.get("primary_category", "N/A"),
        published=chunk.get("published", "N/A"),
        updated=chunk.get("updated", "N/A"),
        references=chunk.get("references", "N/A"),
    )

    # Display the HTML in an IPython notebook
    display(HTML(html_output))


def display_retrieved_context(context_response):
    # HTML template for the main container and individual tables
    html_template = """
    <html>
    <head>
    <style>
        .container {{
            display: flex;
            flex-wrap: wrap;
        }}
        .table-container {{
            margin: 10px;
            padding: 10px;
            border: 1px solid #dddddd;
        }}
        table {{
            font-family: arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
        }}
        td, th {{
            border: 1px solid #dddddd;
            text-align: left;
            padding: 8px;
        }}
    </style>
    </head>
    <body>
        <div class="container">
            {tables}
        </div>
    </body>
    </html>
    """

    # Function to generate HTML table for a single dictionary
    def generate_table_for_dict(data, score):

        def get_value(key, value):
            if value is None:
                return "N/A"
            if key == "authors":
                return value[:500]
            return value

        rows = "\n".join(
            "<tr><td>{key}</td><td>{value}</td></tr>".format(
                key=key, value=value if value is not None else "N/A"
            )
            for key, value in data.items() if key != "authors"
        )
        table_html = """
        <div class="table-container">
            <table>
                <tr>
                    <th>Key</th>
                    <th>Value</th>
                </tr>
                <tr>
                    <td>Similarity Score</td>
                    <td>{score}</td>
                </tr>
                {rows}
            </table>
        </div>
        """.format(
            rows=rows,
            score=score
        )
        return table_html

    # Generate HTML tables for all dictionaries in the list
    tables = "\n".join(
        generate_table_for_dict(data["metadata"], data.score) for data in context_response
    )

    # Format the main HTML with the generated tables
    html_output = html_template.format(tables=tables)

    # Display the HTML in an IPython notebook
    display(HTML(html_output))


def display_markdown(content: str) -> None:
    display(Markdown(content))

![image.png](imgs/visual-breakpoint.png)

![step-away-rag.png](imgs/step-away-rag.png)

## Setup OpenAI

Enter the OpenAI API key and instantiate the OpenAI clinet.

Copy this openai API key into the prompt.

The key can be found here: https://docs.google.com/document/d/1rh-YQ12zNEqmmNWiA_W3PlCxy5HUWiyAr7OS5xQp9sw

In [39]:
import getpass
from openai import OpenAI

OPENAI_API_KEY = getpass.getpass("Enter your OpenAI API key: ")
openai = OpenAI(api_key=OPENAI_API_KEY)

![section-breakpoint.png](imgs/section-breakpoint.png)

## Generate text using OpenAI GPT

In [40]:
def generate(prompt: str, openai_client: OpenAI, model: str = "gpt-4o-mini") -> str:

    # API reference: https://platform.openai.com/docs/api-reference/chat/create
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Output is markdown"},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
    )

    return response.choices[0].message.content


![section-breakpoint.png](imgs/section-breakpoint.png)

#### Now, we can call LLM with and prompt a query to it, let's see how it works

In [41]:
display_markdown(generate("What is the capital of Netherlands?", openai))

The capital of the Netherlands is Amsterdam.

#### So far so good, but let's try with this example

In [43]:
display_markdown(
    generate(
        "What are first words Neil Armstrong said when he landed on Moon?", openai, model="gpt-4o-mini"
    )
)

When Neil Armstrong first set foot on the Moon on July 20, 1969, he famously said, "That's one small step for [a] man, one giant leap for mankind." The phrase is often quoted, and the "a" in the quote has been a topic of discussion, as it was not clearly heard in the transmission.

![section-breakpoint.png](imgs/section-breakpoint.png)

#### What about the knowledge **after** model is trained?

In [44]:
display_markdown(generate("What are key benefits of LLama 3?", openai, model="gpt-4o-mini"))

LLaMA 3, the latest iteration in the LLaMA series of language models developed by Meta, offers several key benefits that enhance its usability and performance. Here are some of the notable advantages:

### 1. **Improved Performance**
   - **Higher Accuracy**: LLaMA 3 is designed to provide more accurate and contextually relevant responses compared to its predecessors.
   - **Better Understanding of Context**: Enhanced capabilities in understanding and maintaining context over longer conversations.

### 2. **Scalability**
   - **Multiple Model Sizes**: LLaMA 3 is available in various sizes, allowing users to choose a model that fits their computational resources and application needs.
   - **Efficient Resource Utilization**: Optimized for performance, making it suitable for deployment in various environments, from cloud to edge devices.

### 3. **Versatility**
   - **Wide Range of Applications**: Suitable for diverse applications, including chatbots, content generation, translation, and more.
   - **Adaptability**: Can be fine-tuned for specific tasks or industries, enhancing its effectiveness in specialized applications.

### 4. **Enhanced Safety and Ethical Considerations**
   - **Mitigated Bias**: Efforts have been made to reduce biases in the model's responses, promoting fairer and more balanced outputs.
   - **Content Moderation**: Improved mechanisms for filtering harmful or inappropriate content, making it safer for users.

### 5. **User-Friendly Features**
   - **Interactive Capabilities**: Designed to engage in more natural and human-like conversations, improving user experience.
   - **Customizability**: Users can customize the model's behavior and personality to better align with their specific needs.

### 6. **Research and Community Support**
   - **Open Research**: Meta continues to support open research initiatives, allowing the community to contribute to and benefit from advancements in AI.
   - **Documentation and Resources**: Comprehensive documentation and resources are available to help users effectively implement and utilize the model.

### 7. **Integration with Other Technologies**
   - **Compatibility**: Can be integrated with various software and platforms, enhancing its utility in existing workflows.
   - **APIs and Tools**: Availability of APIs and development tools to facilitate easy integration and deployment.

### Conclusion
LLaMA 3 represents a significant advancement in language model technology, offering improved performance, versatility, and safety features. Its design caters to a wide range of applications, making it a valuable tool for developers, researchers, and businesses alike.

![section-breakpoint.png](imgs/section-breakpoint.png)

#### Let's feed the context with relevant data

In [46]:
context_example = """
Answer the question based on the following context in markdown format. If you don't can't find the answer, tell I don't know.

Context:
The Llama 3 Herd of Models
Llama Team, AI @ Meta1
1A detailed contributor list can be found in the appendix of this paper.
Modern artificial intelligence (AI) systems are powered by foundation models.
This paper presents a new set of foundation models, called Llama 3.
It is a herd of language models that natively support multilinguality, coding, reasoning, and tool usage.
Our largest model is a dense Transformer with 405B parameters and a context window of up to 128K tokens.
This paper presents an extensive\nempirical evaluation of Llama 3.
We find that Llama 3 delivers comparable quality to leading language
models such as GPT-4 on a plethora of tasks. We publicly release Llama 3,
including pre-trained and post-trained versions of the 405B parameter language model and our
Llama Guard 3 model for input and output safety.
The paper also presents the results of experiments in which we integrate image, video, and speech capabilities into Llama 3
via a compositional approach. We observe this approach
performs competitively with the state-of-the-art on image, video, and speech recognition tasks.

Question: What are key benefits of LLama 3?
"""
display_markdown(generate(context_example, openai))

# Key Benefits of Llama 3

1. **Multilingual Support**: Llama 3 natively supports multiple languages, making it versatile for global applications.

2. **Advanced Capabilities**: It excels in coding, reasoning, and tool usage, enhancing its utility in various domains.

3. **Large Model Size**: The largest model has 405 billion parameters, allowing for complex understanding and generation of language.

4. **Extended Context Window**: With a context window of up to 128K tokens, Llama 3 can handle longer inputs and maintain context over extended interactions.

5. **Competitive Performance**: Llama 3 delivers quality comparable to leading models like GPT-4 across a wide range of tasks.

6. **Safety Features**: The inclusion of Llama Guard 3 ensures input and output safety, addressing concerns related to AI-generated content.

7. **Integration of Modalities**: The model can integrate image, video, and speech capabilities, performing competitively in recognition tasks across these modalities.

![visual-breakpoint.png](imgs/visual-breakpoint.png)

![why-rag.png](imgs/why-rag.png)

![section-breakpoint.png](imgs/section-breakpoint.png)

![how-to-build-rag.png](imgs/how-to-build-rag.png)

![visual-breakpoint.png](imgs/visual-breakpoint.png)

## 1. Build a knowledge base

![section-breakpoint.png](imgs/section-breakpoint.png)

![build-knowledge-base.png](imgs/build-knowledge-base.png)


![section-breakpoint.png](imgs/section-breakpoint.png)

### Setup Pinecone 

Link: https://www.pinecone.io/

Enter the Pinecone API key inside the prompt and create a Pinecone client.

In [47]:
PINECONE_REGION = "us-east-1"
PINECONE_CLOUD = "aws"
INDEX_NAME = "pinecone-workshop-1"
VECTOR_DIMENSIONS = 1536
PINECONE_API_KEY = getpass.getpass("Enter your Pinecone API key: ")

In [48]:
from pinecone.grpc import PineconeGRPC
#from pinecone import Pinecone

pinecone = PineconeGRPC(api_key=PINECONE_API_KEY)
# pinecone = Pinecone(api_key=PINECONE_API_KEY)

pinecone.list_indexes()

{'indexes': [{'dimension': 1536,
              'host': 'pinecone-worshop-1-2kw20wn.svc.apw5-4e34-81fa.pinecone.io',
              'metric': 'cosine',
              'name': 'pinecone-worshop-1',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-west-2'}},
              'status': {'ready': True, 'state': 'Ready'}},
             {'dimension': 1536,
              'host': 'pinecone-workshop-1-2kw20wn.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'pinecone-workshop-1',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}

![section-breakpoint.png](imgs/section-breakpoint.png)

### Create a Pinecone Index

More info o serverless: https://docs.pinecone.io/reference/architecture/serverless-architecture#overview  
Ref API: https://docs.pinecone.io/guides/indexes/create-an-index

In [49]:
from pinecone import ServerlessSpec

# Check if the index already exists and delete it
if INDEX_NAME in [index.name for index in pinecone.list_indexes()]:
    pinecone.delete_index(INDEX_NAME)

# Create a new index with the specified name, dimension, metric, and spec
# Docs: https://docs.pinecone.io/reference/api/control-plane/create_index
pinecone.create_index(
    name=INDEX_NAME,
    dimension=VECTOR_DIMENSIONS,
    metric="cosine",
    spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
)

# Create a new Index reference object with the specified name
index = pinecone.Index(INDEX_NAME)
print(index.describe_index_stats())

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 0}},
 'total_vector_count': 0}


![section-breakpoint.png](imgs/section-breakpoint.png)

### Dataset

We are going to use a sample of 1000 AI papers that can be found here: https://huggingface.co/datasets/smartcat/ai-arxiv2-chunks-embedded 

The data set is already chunked and encoded using `text-embeddings-3-small` so we can just load and upsert it to the Pinecone.

If you want to play with chunking strategies and embeddings, you can find the full data set here: https://huggingface.co/datasets/jamescalam/ai-arxiv2

Dataset API reference: https://huggingface.co/docs/datasets/en/index  
Slicing and indexing: https://huggingface.co/docs/datasets/en/access 

In [50]:
import datasets

dataset = datasets.load_dataset("smartcat/ai-arxiv2-chunks-embedded", split="train")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['doi', 'chunk-id', 'chunk', 'id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 'primary_category', 'published', 'updated', 'references', 'metadata', 'embeddings'],
    num_rows: 80147
})

In [51]:
chunk_display_html(dataset[0])

Key,Value
Title,The Llama 3 Herd of Models
doi,2407.21783
Chunk ID,0
chunk,"The Llama 3 Herd of Models Llama Team, AI @ Meta1 1A detailed contributor list can be found in the appendix of this paper. Modern artificial intelligence (AI) systems are powered by foundation models. This paper presents a new set of foundation models, called Llama 3. It is a herd of language models that natively support multilinguality, coding, reasoning, and tool usage. Our largest model is a dense Transformer with 405B parameters and a context window of up to 128K tokens. This paper presents an extensive empirical evaluation of Llama 3. We find that Llama 3 delivers comparable quality to leading language models such as GPT-4 on a plethora of tasks. We publicly release Llama 3, including pre-trained and post-trained versions of the 405B parameter language model and our Llama Guard 3 model for input and output safety. The paper also presents the results of experiments in which we integrate image, video, and speech capabilities into Llama 3 via a compositional approach. We observe this approach performs competitively with the state-of-the-art on image, video, and speech recognition tasks. The resulting models are not yet being broadly released as they are still under development. Date:July 23, 2024 Website: https://llama.meta.com/ 1 Introduction"
id,2407.21783#0
Summary,"Modern artificial intelligence (AI) systems are powered by foundation models. This paper presents a new set of foundation models, called Llama 3. It is a herd of language models that natively support multilinguality, coding, reasoning, and tool usage. Our largest model is a dense Transformer with 405B parameters and a context window of up to 128K tokens. This paper presents an extensive empirical evaluation of Llama 3. We find that Llama 3 delivers comparable quality to leading language models such as GPT-4 on a plethora of tasks. We publicly release Llama 3, including pre-trained and post-trained versions of the 405B parameter language model and our Llama Guard 3 model for input and output safety. The paper also presents the results of experiments in which we integrate image, video, and speech capabilities into Llama 3 via a compositional approach. We observe this approach performs competitively with the state-of-the-art on image, video, and speech recognition tasks. The resulting models are not yet being broadly released as they are still under development."
Source,http://arxiv.org/pdf/2407.21783
Authors,"Abhimanyu Dubey,Abhinav Jauhri,Abhinav Pandey,Abhishek Kadian,Ahmad Al-Dahle,Aiesha Letman,Akhil Mathur,Alan Schelten,Amy Yang,Angela Fan,Anirudh Goyal,Anthony Hartshorn,Aobo Yang,Archi Mitra,Archie Sravankumar,Artem Korenev,Arthur Hinsvark,Arun Rao,Aston Zhang,Aurelien Rodriguez,Austen Gregerson,Ava Spataru,Baptiste Roziere,Bethany Biron,Binh Tang,Bobbie Chern,Charlotte Caucheteux,Chaya Nayak,Chloe Bi,Chris Marra,Chris McConnell,Christian Keller,Christophe Touret,Chunyang Wu,Corinne Wong,Cristi"
Categories,"cs.AI,cs.CL,cs.CV"
Comment,None


In [52]:
print(len(dataset[0]["embeddings"]))
print(dataset[0]["embeddings"][:10])

1536
[0.019791821, -0.03444159, 0.029808648, -0.035688918, -0.0043083807, -0.01433157, -0.004995685, 0.05595167, -0.068068594, 0.020071834]


In [53]:
list(dataset[0]["metadata"].keys())

['authors',
 'chunk_id',
 'doc_id',
 'primary_category',
 'published',
 'source',
 'summary',
 'text',
 'title',
 'year']

![section-breakpoint.png](imgs/section-breakpoint.png)

### What has been done with data set?

![chunking-dataset.png](imgs/chunking-dataset.png)

![section-breakpoint.png](imgs/section-breakpoint.png)

#### Chunking Strategies
1. Character split (with overlapping)
2. Recursive character split
3. Document specific splitting
4. Semantic Chunking
5. Agentic?
6. More?

Introduction to chunking: https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb

![section-breakpoint.png](imgs/section-breakpoint.png)

### Data upsert to Pinecone

Let insert data to the Pinecone in batches. 

From our data set we need 3 columns:
1. `id` - the ID of the chunk we want to insert
2. `embeddings` - contains a vector embedding of the chunk. It uses `text-embeddings-3-small`
3. `metadata` - a dictinary with additional data about the chunk. 

The code for upserting:
```python
index.upsert(vectors=[ 
    (id1, vector1, metadata1),
    (id2, vector2, metadata2),
    ....
 ])
```
We can also upsert data directly from the `pandas.Dataframe` using `index.upsert_from_dataframe`.  
It requires 3 columns: 
1. `values` - a column that contain a embeddings for each chunk/record
2. `id` - an ID of a record
3. `metadata` - a key-value pairs for record metadata

Upsert from dataframe: https://docs.pinecone.io/guides/data/use-public-pinecone-datasets#upsert-a-dataset-as-a-dataframe 
Avoid quotas and limits: https://docs.pinecone.io/reference/quotas-and-limits  
For scale-up and optimizations make sure to read: : https://docs.pinecone.io/guides/operations/performance-tuning#increasing-throughput  
Metadata filtering: Metadata filtering: https://docs.pinecone.io/guides/data/filter-with-metadata 

In [54]:
from pinecone.grpc import GRPCIndex as Index

def upsert_batch(ds: datasets.Dataset, index: Index, batch_size: int = 200) -> None:
    df = ds.to_pandas()
    df = df[["id", "embeddings", "metadata"]]
    df.rename(columns={"embeddings": "values"}, inplace=True)
    index.upsert_from_dataframe(df, batch_size=batch_size, show_progress=True)

In [55]:
upsert_batch(dataset.select(range(500)), index, batch_size=100)

sending upsert requests:   0%|          | 0/500 [00:00<?, ?it/s]

collecting async responses:   0%|          | 0/5 [00:00<?, ?it/s]

In [56]:
print(index.describe_index_stats())

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 500}},
 'total_vector_count': 500}


In [57]:
upsert_batch(dataset, index, batch_size=100)

sending upsert requests:   0%|          | 0/80147 [00:00<?, ?it/s]

collecting async responses:   0%|          | 0/802 [00:00<?, ?it/s]

![visual-breakpoint.png](imgs/visual-breakpoint.png)

## Revisit Agenda


![done-next.png](imgs/done-next.png)


![visual-breakpoint.png](imgs/visual-breakpoint.png)

## 2. Retrieve against the query



![section-breakpoint.png](imgs/section-breakpoint.png)

![semantic-search.png](imgs/semantic-search.png)

![section-breakpoint.png](imgs/section-breakpoint.png)

Once we inserted everythong to Pinecone, let's query it. The input is query text and the output is the top similar chunks.
Steps: 
1. Encode the input text to generate embeddings 
2. Call Pinecone's query function to retrieve top K similar results

In [58]:
from typing import List


def encode(
    text: str, openai_client: OpenAI, model: str = "text-embedding-3-small"
) -> List[float]:
    # Use the OpenAI API to encode the text
    # Docs: https://platform.openai.com/docs/api-reference/embeddings
    response = openai_client.embeddings.create(
        model=model, input=text
    )

    return response.data[0].embedding

In [59]:
res = encode("What are key benefits of LLama 3?", openai)
print(res[:10])
print(len(res))

[-0.005354322958737612, -0.023311149328947067, 0.0483490489423275, -0.029883945360779762, 0.016752276569604874, -0.004884340334683657, -0.018868939951062202, 0.048209793865680695, 0.0228237584233284, -0.0058068991638720036]
1536


![section-breakpoint.png](imgs/section-breakpoint.png)

### Query Pinecone

In [60]:
from pinecone import QueryResponse


def retrieve(
    query: str, index: Index, openai_client: OpenAI, top_k: int = 10
) -> QueryResponse:
    # Encode the query using the OpenAI API
    query_embedding = encode(query, openai_client)

    # Use the Pinecone index to query the encoded vector
    # Docs: https://docs.pinecone.io/guides/data/query-data
    ret = index.query(
        vector=query_embedding, top_k=top_k, include_metadata=True, include_values=False
    )
    return ret

In [65]:
res = retrieve("What is LLama 3?", index, openai)

display_retrieved_context(res.matches[:2])

Key,Value
Similarity Score,0.65093094
text,"NIH/Multi-needle 98.8 ––97.5––98.1 – 100.0 100.0 90.8 Multilingual MGSM (0-shot, CoT) 68.9 53.229.9 86.971.151.4 91.6 –85.990.5 91.6 Table 2 Performance of finetuned Llama 3 models on key benchmark evaluations. The table compares the performance of the 8B, 70B, and 405B versions of Llama 3 with that of competing models. We boldface the best-performing model in each of three model-size equivalence classes.△Results obtained using 5-shot prompting (no CoT).◁Results obtained without CoT.♢Results obtained using zero-shot prompting. 2 General Overview The model architecture of Llama 3 is illustrated in Figure 1. The development of our Llama 3 language models comprises two main stages: •Language model pre-training. We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training"
chunk_id,9.0
year,2024.0
primary_category,cs.AI
source,http://arxiv.org/pdf/2407.21783
summary,"Modern artificial intelligence (AI) systems are powered by foundation models. This paper presents a new set of foundation models, called Llama 3. It is a herd of language models that natively support multilinguality, coding, reasoning, and tool usage. Our largest model is a dense Transformer with 405B parameters and a context window of up to 128K tokens. This paper presents an extensive empirical evaluation of Llama 3. We find that Llama 3 delivers comparable quality to leading language models such as GPT-4 on a plethora of tasks. We publicly release Llama 3, including pre-trained and post-trained versions of the 405B parameter language model and our Llama Guard 3 model for input and output safety. The paper also presents the results of experiments in which we integrate image, video, and speech capabilities into Llama 3 via a compositional approach. We observe this approach performs competitively with the state-of-the-art on image, video, and speech recognition tasks. The resulting models are not yet being broadly released as they are still under development."
published,20240731.0
title,The Llama 3 Herd of Models
doc_id,2407.21783


![visual-breakpoint.png](imgs/visual-breakpoint.png)

## 3. Augment and generate

![section-breakpoint.png](imgs/section-breakpoint.png)

![workflow-rag-simple.png](imgs/workflow-rag-simple.png)

![section-breakpoint.png](imgs/section-breakpoint.png)

### Generate a final response

Combine all pieces together:
1. Perform a semantic search on input query
2. Build a context (prompt) for a LLM
3. Call LLM to generate a final response
4. Return a final response and retrieved context

The relevant context can be found in metadata, you can use:
1. `title` - a Paper title
2. `published` - a publish date
3. `primart_category` 
4. `summary` - a paper summary
5. `text` - a chunk text - this is the most useful info to build a context for LLM

In [66]:
from typing import Tuple

def from_metadata(metadata: Dict) -> str:
    return f"""
***
    Title: {metadata['title']}
    Authors: {metadata['authors'][:3]}
    Published: {metadata['published']}
    Paper summary: {metadata['summary']}
    Text: {metadata['text']}
    Source: {metadata['source']}

"""


def augment(query: str, query_results: QueryResponse) -> str:
    context = "\n".join(
        [from_metadata(result.metadata) for result in query_results.matches]
    )
    return f"""
Answer the question based on the following context. If you don't can't find the answer, tell I don't know.
The answers should be clear, easy to understand, complete and comprehensive.

Context:
{context}

Question: {query}
"""


def rag(query: str, index: Index, openai_client: OpenAI, top_k: int = 5) -> Tuple[str, QueryResponse]:
    # 1. [RETRIEVE]: Reuse `semantic_search` function to get the top_k results
    # 2. [AUGMENT]: Use `get_prompt` function to generate the prompt (context + question)
    # 3. [GENERATE]: Use `llm_completion` function to generate the response
    # 4. Return the response and query_results
    query_results = retrieve(query, index, openai_client, top_k=top_k)
    prompt = augment(query, query_results)
    response = generate(prompt, openai_client)
    return response, query_results

In [67]:
answer, context = rag("What are key benefits of LLama 3?", index, openai)
display_markdown(answer)

The key benefits of Llama 3 include:

1. **Multilingual Support**: Llama 3 natively supports multiple languages, making it versatile for global applications.

2. **High Parameter Count**: The largest model has 405 billion parameters, which enhances its ability to understand and generate complex language patterns.

3. **Extended Context Window**: It features a context window of up to 128,000 tokens, allowing it to process and generate longer texts effectively.

4. **Competitive Performance**: Llama 3 delivers comparable quality to leading models like GPT-4 across a wide range of tasks, indicating its robustness and effectiveness.

5. **Safety Features**: The inclusion of the Llama Guard 3 model enhances input and output safety, addressing concerns related to harmful content generation.

6. **Integration of Multimedia Capabilities**: Llama 3 has been tested with image, video, and speech capabilities, performing competitively in these areas, which broadens its application scope.

7. **Improved Helpfulness and Harmlessness**: Compared to its predecessor, Llama 3 offers a better balance between being helpful and harmless, making it safer for users.

8. **Extensive Empirical Evaluation**: The model has undergone rigorous testing against various benchmark datasets, ensuring its reliability and performance.

9. **Public Availability**: Llama 3 is publicly released under an updated community license, allowing wider access for research and development.

10. **Enhanced Data Quality and Scale**: The model benefits from improved data quality and a larger training dataset (15 trillion tokens), which contributes to its performance and understanding of language. 

These benefits position Llama 3 as a significant advancement in the field of foundation models for AI.

In [68]:
display_retrieved_context(context.matches[:2])

Key,Value
Similarity Score,0.64085853
text,"parameters. We evaluate the performance of Llama 3 on a plethora of benchmark datasets that span a wide range of language understanding tasks. In addition, we perform extensive human evaluations that compare Llama 3 with competing models. An overview of the performance of the flagship Llama 3 model on key benchmarks is presented in Table 2. Our experimental evaluation suggests that our flagship model performs on par with leading language models such as GPT-4 (OpenAI, 2023a) across a variety of tasks, and is close to matching the state-of-the-art. Our smaller models are best-in-class, outperforming alternative models with similar numbers of parameters (Bai et al., 2023; Jiang et al., 2023). Llama 3 also delivers a much better balance between helpfulness and harmlessness than its predecessor (Touvron et al., 2023b). We present a detailed analysis of the safety of Llama 3 in Section 5.4. We are publicly releasing all three Llama 3 models under an updated version of the Llama 3 Community License; seehttps://llama.meta.com . This includes pre-trained and post-trained versions of our 405B parameter language model and a new version of our Llama Guard model (Inan et al., 2023) for input and output safety."
chunk_id,5.0
year,2024.0
primary_category,cs.AI
source,http://arxiv.org/pdf/2407.21783
summary,"Modern artificial intelligence (AI) systems are powered by foundation models. This paper presents a new set of foundation models, called Llama 3. It is a herd of language models that natively support multilinguality, coding, reasoning, and tool usage. Our largest model is a dense Transformer with 405B parameters and a context window of up to 128K tokens. This paper presents an extensive empirical evaluation of Llama 3. We find that Llama 3 delivers comparable quality to leading language models such as GPT-4 on a plethora of tasks. We publicly release Llama 3, including pre-trained and post-trained versions of the 405B parameter language model and our Llama Guard 3 model for input and output safety. The paper also presents the results of experiments in which we integrate image, video, and speech capabilities into Llama 3 via a compositional approach. We observe this approach performs competitively with the state-of-the-art on image, video, and speech recognition tasks. The resulting models are not yet being broadly released as they are still under development."
published,20240731.0
title,The Llama 3 Herd of Models
doc_id,2407.21783


![section-breakpoint.png](imgs/section-breakpoint.png)

In [69]:
answer, context = rag("What is skeleton of thoughts?", index, openai)
display_markdown(answer)

The "Skeleton of Thoughts" (SoT) is a method proposed to improve the efficiency and quality of responses generated by large language models (LLMs). It involves two main stages:

1. **Skeleton Stage**: The LLM first generates a concise outline or "skeleton" of the answer to a given question. This skeleton serves as a structured framework for the response.

2. **Point-Expanding Stage**: After the skeleton is created, the LLM expands on each point of the skeleton in parallel, rather than sequentially. This parallel processing allows for faster generation of the complete answer.

The SoT approach is inspired by how humans typically think and write, organizing their thoughts before elaborating on them. It has been shown to provide significant speed-ups in response times and can enhance the quality of answers across various question categories.

In [79]:
answer, context = rag("How to apply LLMs to recommender engines?", index, openai)
display_markdown(answer)

To apply Large Language Models (LLMs) to recommender systems, researchers have proposed several methods and approaches that can be categorized into different strategies:

1. **LLMs as Recommendation Models**:
   - **Zero-shot Recommendations**: Some methods utilize LLMs in a zero-shot paradigm, where the model is prompted to complete recommendation tasks without any parameter tuning. Techniques like prompt engineering, including recency-focused and in-context learning, are employed to enhance recommendation performance and mitigate potential biases.
   - **Instruction Tuning**: Another approach involves specializing LLMs for personalized recommendations through instruction tuning. This requires high-quality instruction data, which can be constructed from user-item interactions using heuristic templates. The InstructRec technique, for example, simulates diverse user instructions to improve the model's adaptability to various recommendation scenarios.

2. **LLM-enhanced Recommendation Models**:
   - **Inferring User Intentions**: LLMs can be used to infer users' potential intentions from their historical interaction data. Traditional recommendation models can then utilize these inferred intentions to improve the retrieval of relevant items.
   - **Feature Encoding**: LLMs can serve as feature encoders, processing side information about items and users (like item descriptions and user reviews) to create more informative representations. These representations are then integrated into traditional recommender systems as augmented input.
   - **Distillation Techniques**: Some studies adopt a distillation-like approach to transfer the capabilities of LLMs to smaller traditional recommendation models. This involves aligning the hidden states of LLMs with those of smaller models through joint training, allowing for efficient deployment without the overhead of using large models online.

3. **LLMs as Recommendation Simulators**:
   - LLMs can also be utilized to simulate user-item interactions, helping to generate recommendations based on learned patterns from user behavior and item characteristics.

4. **Addressing Challenges**:
   - Despite the potential of LLMs, there are challenges such as the semantic gap between natural language and recommendation tasks, cold-start problems, and the need for efficient inference. Techniques like efficient tuning, quantization, and improved context modeling are essential for deploying LLMs effectively in real-world recommender systems.

By leveraging these strategies, LLMs can enhance the capabilities of recommender systems, improving their performance and user experience.

![section-breakpoint.png](imgs/section-breakpoint.png)

### What about comparisons?

In [74]:
answer, context = rag(
    "Write side by side summaries between mistral, kosmos, palm and llama 3. If there is key difference between them, what is it",
    index,
    openai,
    top_k=10,
)
display_markdown(answer)

Here is a side-by-side summary of the Mistral 7B, Kosmos-1, Llama 3, and PaLM models, highlighting their key features and differences:

| Feature/Model       | **Mistral 7B**                                                                 | **Kosmos-1**                                                                 | **Llama 3**                                                                 | **PaLM** (not detailed in the context) |
|---------------------|--------------------------------------------------------------------------------|-------------------------------------------------------------------------------|----------------------------------------------------------------------------|-----------------------------------------|
| **Parameters**      | 7 billion                                                                       | Not specified, but designed as a multimodal model                           | Up to 405 billion                                                            | Not specified                           |
| **Architecture**    | Uses grouped-query attention (GQA) and sliding window attention (SWA)         | Multimodal Large Language Model (MLLM) capable of processing text and images | Dense Transformer architecture                                               | Not specified                           |
| **Performance**     | Outperforms Llama 2 13B across all benchmarks, especially in reasoning, math, and code generation | Strong performance in language understanding, generation, and multimodal tasks | Comparable quality to leading models like GPT-4 across various tasks        | Not specified                           |
| **Instruction Tuning** | Includes a fine-tuned version (Mistral 7B - Instruct) that excels in following instructions | Trained to follow instructions in zero-shot and few-shot settings            | Supports instruction tuning and fine-tuning for various tasks               | Not specified                           |
| **Use Cases**       | Effective for a wide range of applications, including chat and instruction following | Designed for multimodal tasks, including image captioning and visual question answering | Supports multilinguality, coding, reasoning, and tool usage                 | Not specified                           |
| **Release License** | Released under Apache 2.0 license                                             | Not specified                                                                 | Publicly released, including pre-trained and post-trained versions          | Not specified                           |
| **Key Differences**  | Focuses on efficiency and performance in language tasks with a smaller model size | Emphasizes multimodal capabilities, integrating text and image processing     | Largest model with extensive capabilities across various tasks               | Not specified                           |

### Key Differences:
- **Mistral 7B** is specifically engineered for high performance and efficiency in language tasks, utilizing advanced attention mechanisms while maintaining a smaller parameter count.
- **Kosmos-1** stands out as a multimodal model, capable of processing both text and images, which is a significant shift from traditional language models that focus solely on text.
- **Llama 3** is notable for its large parameter size (up to 405 billion) and its ability to support a wide range of tasks, including multilingual capabilities and tool usage.
- **PaLM** is not detailed in the provided context, but it is generally known for its large-scale language processing capabilities.

In summary, the key differences lie in their architectural focus (text vs. multimodal), parameter sizes, and specific use cases, with Mistral 7B emphasizing efficiency, Kosmos-1 focusing on multimodal tasks, and Llama 3 offering extensive capabilities with a larger model size.

In [75]:
display_retrieved_context(context.matches[:3])

Key,Value
Similarity Score,0.54317
text,"Detailed results for Mistral 7B, Llama 2 7B/13B, and Code-Llama 7B are reported in Table 2. Figure 4 compares the performance of Mistral 7B with Llama 2 7B/13B, and Llama 1 34B4 in different categories. Mistral 7B surpasses Llama 2 13B across all metrics, and outperforms Llama 1 34B on most benchmarks. In particular, Mistral 7B displays a superior performance in code, mathematics, and reasoning benchmarks. 4Since Llama 2 34B was not open-sourced, we report results for Llama 1 34B. 3 jm Mistral 7B = mm LLaMA2 138 50 lm Mistral 7B mm LLaMA2 138 mmm LlaMA278 lm LLaMA1 348 bel mmm LlaMA2 78 mem LlaMA 1348 70 40 vt = = eo g 7 = 330 Â£ g gs0 : < <20 40 10 ay MMLU Knowledge Reasoning Comprehension AGI Eval Math BBH Code"
chunk_id,9.0
year,2023.0
primary_category,cs.CL
source,http://arxiv.org/pdf/2310.06825
summary,"We introduce Mistral 7B v0.1, a 7-billion-parameter language model engineered for superior performance and efficiency. Mistral 7B outperforms Llama 2 13B across all evaluated benchmarks, and Llama 1 34B in reasoning, mathematics, and code generation. Our model leverages grouped-query attention (GQA) for faster inference, coupled with sliding window attention (SWA) to effectively handle sequences of arbitrary length with a reduced inference cost. We also provide a model fine-tuned to follow instructions, Mistral 7B -- Instruct, that surpasses the Llama 2 13B -- Chat model both on human and automated benchmarks. Our models are released under the Apache 2.0 license."
published,20231010.0
title,Mistral 7B
doc_id,2310.06825


![visual-breakpoint.png](imgs/visual-breakpoint.png)

## Hands-On Exercises

### Exercise #1 - Query Comprehension

Your task is to figure out how to execute the following query:  

`Write side by side summaries between mistral, kosmos, palm and llama 3. If there is key difference between them, what is it`

Hints:
1. JSON response from GPT: https://platform.openai.com/docs/api-reference/chat/create 
2. Use LLM to extract information from the query
3. Apply the right techniques of RAG to construct the best response


In [37]:
# TODO: Implement a query comprehension tool



### Exercise #2 - Paper listing and filtering

Adapt RAG model to work on paper listing, ie it should be able to answer this type of question:  
`List me papers names and sources that are published in 2023 about e-commerce search techniques`

Hints:
1. Use metadata filtering: https://docs.pinecone.io/guides/data/filter-with-metadata (`metadata_filter={"published": {"gt": 20230101}}`)
2. Use LLM (prompt engineering) to extract the useful data from the query (search term, date, task, etc)
3. JSON reponse from ChatGPT - https://platform.openai.com/docs/api-reference/chat/create   

In [36]:
# TODO: Implement paper listing and metadata filtering

## Exercise #3 - Reranking

Adapt RAG model to provide more quality answers by applying reranking step. 

![pinecone-reranking.png](imgs/pinecone-rerank.png)


Hints:
1. Fetch more chunks (top_k>50)
2. Rerank them using model rankers (https://huggingface.co/BAAI/bge-reranker-base or you can use LLM as reranker)
3. Send TOP 3 chunk to generate step to answer the questions
4. Compare results with ranking and without reranking
5. Experiment with different context feed for LLM generate step

Hints 2:
1. Use Reranking API from Pinecone: https://docs.pinecone.io/models/bge-reranker-v2-m3 

Example queries:
1. What are the best practices for designing prompts in promptable segmentation tasks to maximize model performance on diverse downstream applications?
2. How can prompt ambiguity be managed in segmentation models to ensure valid and useful output masks?
3. How do prompt-based techniques in computer vision compare to those in natural language processing for adapting to new tasks?
4. Why would I need RLHF?



In [ ]:
# TODO: Implement reraanking of the results

![visual-breakpoint.png](imgs/visual-breakpoint.png)

# Appendix: Fine-Tune vs RAG vs Prompt Engineering

![rag-vs-prompt-vs-finetune.png](imgs/finetune-vs-rag.png)

![visual-breakpoint.png](imgs/visual-breakpoint.png)